In [4]:
from collections import deque
import heapq
import os
import time
import tracemalloc
import pandas as pd

# --- 1. VERIFICACIÓN Y CARGA DE DATOS ---
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

target_file = 'data/ecommerce_1K.parquet'
print(f'📁 Cargando dataset de prueba desde: {target_file}')
df = pd.read_parquet(target_file)
print(
    f'✅ Dataset cargado con éxito. Total de registros: {len(df):,}\n'
    '------------------------------------------------------------'
)

📁 Cargando dataset de prueba desde: data/ecommerce_1K.parquet
✅ Dataset cargado con éxito. Total de registros: 1,000
------------------------------------------------------------


In [15]:
# ============================================================
# ETAPA 0: Ingesta y Búfer Continuo (Stream a Memoria)
# ============================================================
print('--- [ETAPA 0] Ingesta y Búfer FIFO (Queue) ---')

# Iniciamos monitoreo de memoria y tiempo
tracemalloc.start()
start_time = time.perf_counter()

# Simulamos la estructura de Cola FIFO acotada para el dataset completo
BUFFER_CAPACITY = 5000
fifo_buffer = deque(maxlen=BUFFER_CAPACITY)

# Inyectamos todas las filas del DataFrame al búfer
for _, row in df.iterrows():
    fifo_buffer.append(row.to_dict())

# Extraemos TODO el lote crudo disponible en el búfer (los 1k elementos)
lote_crudo = [fifo_buffer.popleft() for _ in range(len(fifo_buffer))]

end_time = time.perf_counter()
_, peak_mem_etapa0 = tracemalloc.get_traced_memory()
tracemalloc.stop()

tiempo_etapa0 = (end_time - start_time) * 1000  # Convertir a ms

print(f'    > Elementos en búfer FIFO: {len(fifo_buffer)}')
print(f'    > Lote crudo total extraído: {len(lote_crudo)} elementos')
print(f'    ⏱️ Tiempo de ejecución: {tiempo_etapa0:.4f} ms')
print(
    f'    💾 Memoria RAM pico: {peak_mem_etapa0 / 1024:.2f} KB\n'
    '------------------------------------------------------------'
)

--- [ETAPA 0] Ingesta y Búfer FIFO (Queue) ---
    > Elementos en búfer FIFO: 0
    > Lote crudo total extraído: 1000 elementos
    ⏱️ Tiempo de ejecución: 451.2922 ms
    💾 Memoria RAM pico: 1680.62 KB
------------------------------------------------------------


In [7]:
# ============================================================
# ETAPA 1: Preprocesamiento de Lotes (Limpieza y Deduplicación)
# ============================================================
print('--- [ETAPA 1] Preprocesamiento de Lotes (Pandas / Timsort) ---')

tracemalloc.start()
start_time = time.perf_counter()

# Convertimos el lote crudo a un DataFrame temporal para aplicar limpieza
df_lote = pd.DataFrame(lote_crudo)

# 1. Limpieza de nulos y normalización básica
df_lote = df_lote.dropna(subset=['event_id', 'product_id', 'price'])

# 2. Deduplicación basada en event_id y user_id
df_lote = df_lote.drop_duplicates(subset=['event_id', 'user_id'])

# Convertimos de nuevo a lista de diccionarios limpios
lote_limpio = df_lote.to_dict(orient='records')

end_time = time.perf_counter()
_, peak_mem_etapa1 = tracemalloc.get_traced_memory()
tracemalloc.stop()

tiempo_etapa1 = (end_time - start_time) * 1000

print(f'   > Elementos después de limpieza y deduplicación: {len(lote_limpio)}')
print(f'   ⏱️ Tiempo de ejecución: {tiempo_etapa1:.4f} ms')
print(
    f'   💾 Memoria RAM pico: {peak_mem_etapa1 / 1024:.2f} KB\n'
    '------------------------------------------------------------'
)

--- [ETAPA 1] Preprocesamiento de Lotes (Pandas / Timsort) ---
   > Elementos después de limpieza y deduplicación: 100
   ⏱️ Tiempo de ejecución: 80.8495 ms
   💾 Memoria RAM pico: 137.75 KB
------------------------------------------------------------


In [10]:
# ============================================================
# ETAPA 2: Cola de Prioridades de Negocio (MinHeap)
# ============================================================
print('--- [ETAPA 2] Cola de Prioridades de Negocio (MinHeap) ---')

tracemalloc.start()
start_time = time.perf_counter()

# Definimos una función de prioridad de negocio:
# Por ejemplo, las compras ('purchase') tienen mayor prioridad (número menor = sale primero),
# seguidas de agregar al carrito ('cart'), y vistas ('view').
def calcular_prioridad(event_type):
    prioridades = {'purchase': 1, 'cart': 2, 'view': 3}
    return prioridades.get(event_type, 5)


# Construimos el MinHeap basado en la prioridad de negocio
min_heap = []
for i, evento in enumerate(lote_limpio):
    prio = calcular_prioridad(evento.get('event_type'))
    # CORRECCIÓN: Se incluye 'i' (contador entero único) en la tupla 
    # para evitar que Python intente comparar diccionarios en caso de empate.
    heapq.heappush(
        min_heap, (prio, str(evento.get('event_time', '')), i, evento)
    )

# Extraemos los 5 eventos más críticos ordenados por prioridad de negocio
eventos_prioritarios = [heapq.heappop(min_heap) for _ in range(min(5, len(min_heap)))]

end_time = time.perf_counter()
_, peak_mem_etapa2 = tracemalloc.get_traced_memory()
tracemalloc.stop()

tiempo_etapa2 = (end_time - start_time) * 1000

print(f'   > Total de elementos organizados en el MinHeap: {len(min_heap)}')
print(f'   > Top 5 eventos con mayor prioridad de negocio extraídos:')
for prio, t, i, ev in eventos_prioritarios:
    print(
        f'       - Prioridad [{prio}] | Tipo: {ev.get("event_type")} |'
        f' Precio: ${ev.get("price")}'
    )

print(f'   ⏱️ Tiempo de ejecución: {tiempo_etapa2:.4f} ms')
print(
    f'   💾 Memoria RAM pico: {peak_mem_etapa2 / 1024:.2f} KB'
    '------------------------------------------------------------'
)

--- [ETAPA 2] Cola de Prioridades de Negocio (MinHeap) ---
   > Total de elementos organizados en el MinHeap: 95
   > Top 5 eventos con mayor prioridad de negocio extraídos:
       - Prioridad [2] | Tipo: cart | Precio: $503.09
       - Prioridad [3] | Tipo: view | Precio: $489.07
       - Prioridad [3] | Tipo: view | Precio: $293.65
       - Prioridad [3] | Tipo: view | Precio: $28.31
       - Prioridad [3] | Tipo: view | Precio: $712.87
   ⏱️ Tiempo de ejecución: 4.0909 ms
   💾 Memoria RAM pico: 28.47 KB------------------------------------------------------------
